In [ ]:
# Example 1: Basic Version Determination from Git Repository
import subprocess
import tempfile
import os
from setuptools_git_versioning import version_from_git, get_branch, is_dirty

print("Example 1: Basic Version Determination from Git Repository")
print("=" * 60)

# Create a temporary git repository for testing
with tempfile.TemporaryDirectory() as temp_dir:
    os.chdir(temp_dir)
    
    # Initialize git repository
    subprocess.run(['git', 'init'], capture_output=True, text=True, check=True)
    subprocess.run(['git', 'config', 'user.name', 'Test User'], capture_output=True, check=True)
    subprocess.run(['git', 'config', 'user.email', 'test@example.com'], capture_output=True, check=True)
    
    # Create pyproject.toml with setuptools-git-versioning configuration
    pyproject_content = """[tool.setuptools-git-versioning]
enabled = true
template = "{tag}"
dev_template = "{tag}.dev{ccount}"
dirty_template = "{tag}.dev{ccount}.dirty"
starting_version = "0.1.0"
"""
    with open('pyproject.toml', 'w') as f:
        f.write(pyproject_content)
    
    # Create initial commit
    with open('README.md', 'w') as f:
        f.write("# Test Package\nTest repository for git versioning")
    
    subprocess.run(['git', 'add', '.'], capture_output=True, check=True)
    subprocess.run(['git', 'commit', '-m', 'Initial commit'], capture_output=True, check=True)
    
    # Test version determination from clean repository
    version = version_from_git(temp_dir)
    print(f"✅ Version from clean repo: {version}")
    print(f"   Version type: {type(version)}")
    
    # Get additional git information
    current_branch = get_branch(temp_dir)
    print(f"   Current branch: {current_branch}")
    
    repo_is_dirty = is_dirty(temp_dir)
    print(f"   Repository is dirty: {repo_is_dirty}")
    
    # Get commit SHA using git directly
    commit_sha = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], 
                               capture_output=True, text=True, check=True)
    print(f"   Current commit SHA: {commit_sha.stdout.strip()}")
    
    # Check git status
    git_status = subprocess.run(['git', 'status', '--porcelain'], 
                               capture_output=True, text=True)
    print(f"   Git status clean: {len(git_status.stdout.strip()) == 0}")
    
    # Add a tag and test again
    subprocess.run(['git', 'tag', 'v1.0.0'], capture_output=True, check=True)
    version_with_tag = version_from_git(temp_dir)
    print(f"\n✅ Version after tagging v1.0.0: {version_with_tag}")

print("\n✅ Basic git versioning functionality demonstrated")

## Example 2: Version Generation with Git Tags

In [ ]:
# Example 2: Version Generation with Git Tags
# setuptools-git-versioning 3.x API: templates are keyword-only, the repo is passed
# via root= (NOT a positional arg), and version_from_git returns a packaging Version.
from setuptools_git_versioning import version_from_git, get_all_tags

print("Example 2: Version Generation with Git Tags")

with tempfile.TemporaryDirectory() as temp_dir:
    os.chdir(temp_dir)
    for cmd in (['git', 'init'], ['git', 'config', 'user.name', 'Test User'],
                ['git', 'config', 'user.email', 'test@example.com']):
        subprocess.run(cmd, capture_output=True, check=True)
    with open('setup.py', 'w') as f:
        f.write('from setuptools import setup\nsetup(name="test-package")')
    subprocess.run(['git', 'add', '.'], capture_output=True, check=True)
    subprocess.run(['git', 'commit', '-m', 'Initial setup'], capture_output=True, check=True)
    subprocess.run(['git', 'tag', 'v1.0.0'], capture_output=True, check=True)

    tmpl = {"dev_template": "{tag}.dev{ccount}+{sha}",
            "dirty_template": "{tag}.dev{ccount}+{sha}.dirty"}
    tagged = version_from_git(root=temp_dir, **tmpl)
    print(f"version at tag v1.0.0: {tagged} (type: {type(tagged).__name__})")
    assert str(tagged) == "1.0.0"
    all_tags = get_all_tags(root=temp_dir)
    print(f"all tags: {all_tags}")
    assert all_tags == ["v1.0.0"]

    with open('src.py', 'w') as f:
        f.write('print("hello")')
    subprocess.run(['git', 'add', 'src.py'], capture_output=True, check=True)
    subprocess.run(['git', 'commit', '-m', 'Add source'], capture_output=True, check=True)
    dev_version = str(version_from_git(root=temp_dir, **tmpl))
    print(f"dev version after commit: {dev_version}")
    assert 'dev' in dev_version and '+' in dev_version

    with open('uncommitted.txt', 'w') as f:
        f.write('uncommitted changes')
    subprocess.run(['git', 'add', 'uncommitted.txt'], capture_output=True, check=True)
    dirty_version = str(version_from_git(root=temp_dir, **tmpl))
    print(f"dirty version: {dirty_version}")
    assert 'dirty' in dirty_version

os.chdir(tempfile.gettempdir())
print("Git tag versioning patterns demonstrated")

## Example 3: Custom Configuration and Formatting

In [ ]:
# Example 3: Custom Configuration and Template Formatting
# 3.x API: DEFAULT_* constants live in setuptools_git_versioning.defaults; templates
# are passed as keyword args (version_from_git does not read pyproject.toml itself).
from setuptools_git_versioning import version_from_git
from setuptools_git_versioning.defaults import DEFAULT_CONFIG, DEFAULT_TEMPLATE, DEFAULT_DEV_TEMPLATE

print("Example 3: Custom Configuration and Template Formatting")
print("=" * 60)

# Show default configuration
print("Default configuration:")
for key, value in DEFAULT_CONFIG.items():
    print(f"  {key}: {value}")
print()

print(f"Default template: {DEFAULT_TEMPLATE}")
print(f"Default dev template: {DEFAULT_DEV_TEMPLATE}")
print()

with tempfile.TemporaryDirectory() as temp_dir:
    os.chdir(temp_dir)

    # Initialize repository
    subprocess.run(['git', 'init'], capture_output=True, check=True)
    subprocess.run(['git', 'config', 'user.name', 'Test User'], capture_output=True, check=True)
    subprocess.run(['git', 'config', 'user.email', 'test@example.com'], capture_output=True, check=True)

    # Test different template configurations
    configurations = [
        {
            "name": "Simple semantic versioning",
            "template": "{tag}",
            "dev_template": "{tag}.dev{ccount}",
            "dirty_template": "{tag}.dev{ccount}.dirty"
        },
        {
            "name": "With commit hash",
            "template": "{tag}",
            "dev_template": "{tag}.dev{ccount}+{sha}",
            "dirty_template": "{tag}.dev{ccount}+{sha}.dirty"
        },
        {
            "name": "Custom format with branch",
            "template": "{tag}",
            "dev_template": "{tag}.dev{ccount}+{branch}.{sha}",
            "dirty_template": "{tag}.dev{ccount}+{branch}.{sha}.dirty"
        }
    ]

    # Create initial commit
    with open('README.md', 'w') as f:
        f.write('# Test Package')
    subprocess.run(['git', 'add', 'README.md'], capture_output=True, check=True)
    subprocess.run(['git', 'commit', '-m', 'Initial commit'], capture_output=True, check=True)
    subprocess.run(['git', 'tag', 'v2.0.0'], capture_output=True, check=True)

    # Test each configuration (templates passed as keyword args — the 3.x API)
    for i, config in enumerate(configurations, 1):
        print(f"Configuration {i}: {config['name']}")
        print(f"  Template: {config['template']}")
        print(f"  Dev template: {config['dev_template']}")

        version = str(version_from_git(
            root=temp_dir,
            template=config['template'],
            dev_template=config['dev_template'],
            dirty_template=config['dirty_template'],
        ))
        print(f"  ✅ Version: {version}")
        print()

    # Demonstrate version with additional commits
    with open('feature.py', 'w') as f:
        f.write('def new_feature():\n    pass')
    subprocess.run(['git', 'add', 'feature.py'], capture_output=True, check=True)
    subprocess.run(['git', 'commit', '-m', 'Add feature'], capture_output=True, check=True)

    print("After additional commit (using last config):")
    last = configurations[-1]
    dev_version = str(version_from_git(
        root=temp_dir,
        template=last['template'],
        dev_template=last['dev_template'],
        dirty_template=last['dirty_template'],
    ))
    print(f"  ✅ Development version: {dev_version}")
    print(f"     Shows dev build: {'dev' in dev_version}")
    print(f"     Includes branch: {'main' in dev_version or 'master' in dev_version}")

    # Available template substitutions
    print("\n📝 Available template substitutions:")
    substitutions = [
        "{tag} - Most recent git tag",
        "{ccount} - Commit count since tag",
        "{sha} - Current commit hash (short)",
        "{branch} - Current git branch",
        "{timestamp} - Current timestamp",
        "{env:VAR_NAME} - Environment variable"
    ]
    for sub in substitutions:
        print(f"  • {sub}")

os.chdir(tempfile.gettempdir())
print("\n✅ Custom configuration and template formatting demonstrated")

## Example 4: Branch-Based Versioning and Development Workflow

In [ ]:
# Example 4: Branch-Based Versioning and Count Functions
# 3.x API: branch-aware templates go in as keyword args; str() the Version for checks.
from setuptools_git_versioning import version_from_git, get_branch, get_branches, count_since

print("Example 4: Branch-Based Versioning and Count Functions")
print("=" * 60)

BRANCH_TEMPLATES = {
    "dev_template": "{tag}.dev{ccount}+{branch}",
    "dirty_template": "{tag}.dev{ccount}+{branch}.dirty",
}

with tempfile.TemporaryDirectory() as temp_dir:
    os.chdir(temp_dir)

    # Initialize repository
    subprocess.run(['git', 'init', '-b', 'main'], capture_output=True, check=True)
    subprocess.run(['git', 'config', 'user.name', 'Test User'], capture_output=True, check=True)
    subprocess.run(['git', 'config', 'user.email', 'test@example.com'], capture_output=True, check=True)

    # Create initial commit on main branch
    with open('README.md', 'w') as f:
        f.write('# Main Branch\nProduction code')
    subprocess.run(['git', 'add', '.'], capture_output=True, check=True)
    subprocess.run(['git', 'commit', '-m', 'Initial commit on main'], capture_output=True, check=True)
    subprocess.run(['git', 'tag', 'v1.0.0'], capture_output=True, check=True)

    # Get current branch
    current_branch = get_branch(temp_dir)
    print(f"✅ Current branch: {current_branch}")

    # Test version on main branch
    main_version = str(version_from_git(root=temp_dir, **BRANCH_TEMPLATES))
    print(f"   Main branch version: {main_version}")

    # Get all branches
    all_branches = get_branches(temp_dir)
    print(f"   All branches: {all_branches}")

    # Create and switch to development branch
    subprocess.run(['git', 'checkout', '-b', 'development'], capture_output=True, check=True)

    # Add development commits
    for i in range(3):
        with open(f'feature_{i}.py', 'w') as f:
            f.write(f'# Feature {i}\ndef feature_{i}():\n    return "dev"')
        subprocess.run(['git', 'add', f'feature_{i}.py'], capture_output=True, check=True)
        subprocess.run(['git', 'commit', '-m', f'Add feature {i}'], capture_output=True, check=True)

    # Test branch information
    dev_branch = get_branch(temp_dir)
    print(f"\n✅ Switched to branch: {dev_branch}")

    # Count commits since tag (note: name first, then root)
    commit_count = count_since('v1.0.0', temp_dir)
    print(f"   Commits since v1.0.0: {commit_count}")

    # Test version on development branch
    dev_version = str(version_from_git(root=temp_dir, **BRANCH_TEMPLATES))
    print(f"   Development version: {dev_version}")
    print(f"   Contains branch name: {'development' in dev_version}")
    assert 'development' in dev_version

    # Create feature branch
    subprocess.run(['git', 'checkout', '-b', 'feature/new-api'], capture_output=True, check=True)

    # Add feature commits
    with open('api.py', 'w') as f:
        f.write('# New API\ndef api_endpoint():\n    return {"status": "ok"}')
    subprocess.run(['git', 'add', 'api.py'], capture_output=True, check=True)
    subprocess.run(['git', 'commit', '-m', 'Add new API endpoint'], capture_output=True, check=True)

    # Test feature branch
    feature_branch = get_branch(temp_dir)
    print(f"\n✅ Feature branch: {feature_branch}")

    feature_version = str(version_from_git(root=temp_dir, **BRANCH_TEMPLATES))
    print(f"   Feature version: {feature_version}")

    # Count from original tag
    feature_count = count_since('v1.0.0', temp_dir)
    print(f"   Commits since v1.0.0: {feature_count}")

    # Count commits between HEAD and tag using git directly
    commit_count_git = subprocess.run(['git', 'rev-list', '--count', 'v1.0.0..HEAD'],
                                     capture_output=True, text=True, check=True)
    print(f"   Git rev-list count: {commit_count_git.stdout.strip()}")
    assert feature_count == int(commit_count_git.stdout.strip())

    # Get all branches now
    final_branches = get_branches(temp_dir)
    print(f"\n   All branches in repository: {final_branches}")

    # Switch back to main and create release tag
    subprocess.run(['git', 'checkout', 'main'], capture_output=True, check=True)
    subprocess.run(['git', 'tag', 'v1.1.0-rc1'], capture_output=True, check=True)

    rc_version = str(version_from_git(root=temp_dir, **BRANCH_TEMPLATES))
    print(f"\n✅ Release candidate on main: {rc_version}")

os.chdir(tempfile.gettempdir())
print("\n✅ Branch-based versioning and counting demonstrated")

## Example 5: Production Package Setup and CI/CD Integration

In [ ]:
# Example 5: Production Setup with setup.py Integration
# 3.x API: build-time config still lives in pyproject.toml (read by the setuptools hook);
# for direct calls the same templates are passed as keyword args and str()'d for checks.
from setuptools_git_versioning import version_from_git
import json

print("Example 5: Production Setup with setup.py Integration")
print("=" * 60)

PIPELINE_TEMPLATES = {
    "dev_template": "{tag}.dev{ccount}+{sha}",
    "dirty_template": "{tag}.dev{ccount}+{sha}.dirty",
}

with tempfile.TemporaryDirectory() as temp_dir:
    os.chdir(temp_dir)

    # Initialize production repository
    subprocess.run(['git', 'init', '-b', 'main'], capture_output=True, check=True)
    subprocess.run(['git', 'config', 'user.name', 'CI Bot'], capture_output=True, check=True)
    subprocess.run(['git', 'config', 'user.email', 'ci@company.com'], capture_output=True, check=True)

    # Create comprehensive project structure
    project_files = {
        'pyproject.toml': """[build-system]
requires = ["setuptools>=45", "wheel", "setuptools-git-versioning>=2.0"]
build-backend = "setuptools.build_meta"

[project]
name = "production-package"
dynamic = ["version"]
description = "Production package with automatic git versioning"
requires-python = ">=3.8"

[tool.setuptools-git-versioning]
enabled = true
template = "{tag}"
dev_template = "{tag}.dev{ccount}+{sha}"
dirty_template = "{tag}.dev{ccount}+{sha}.dirty"
starting_version = "0.1.0"
""",
        'setup.py': """from setuptools import setup, find_packages

setup(
    name="production-package",
    packages=find_packages(where="src"),
    package_dir={"": "src"},
)
""",
        'README.md': """# Production Package
Demonstrates setuptools-git-versioning in production.

## Features
- Automatic version from Git tags
- PEP 440 compliant versioning
- CI/CD friendly
""",
        'src/production_package/__init__.py': """\"\"\"Production package with git versioning.\"\"\"

__all__ = ["get_info"]

def get_info():
    return {
        "name": "production-package",
        "git_versioned": True
    }
""",
    }

    # Create all files
    for file_path, content in project_files.items():
        dir_path = os.path.dirname(file_path)
        if dir_path:
            os.makedirs(dir_path, exist_ok=True)
        with open(file_path, 'w') as f:
            f.write(content)

    # Initial commit
    subprocess.run(['git', 'add', '.'], capture_output=True, check=True)
    subprocess.run(['git', 'commit', '-m', 'Initial production setup'], capture_output=True, check=True)

    print("Project structure created:")
    for file_path in project_files.keys():
        print(f"  ✓ {file_path}")

    # Simulate CI/CD pipeline stages
    pipeline_stages = [
        {"stage": "Development", "tag": None, "commits": 2},
        {"stage": "Release Candidate", "tag": "v1.0.0-rc1", "commits": 1},
        {"stage": "Production Release", "tag": "v1.0.0", "commits": 0},
        {"stage": "Hotfix Development", "tag": None, "commits": 1},
        {"stage": "Hotfix Release", "tag": "v1.0.1", "commits": 0}
    ]

    print("\n" + "=" * 60)
    print("CI/CD Pipeline Simulation:")
    print("=" * 60)

    for stage_info in pipeline_stages:
        stage = stage_info["stage"]
        tag = stage_info["tag"]
        commits = stage_info["commits"]

        # Add commits if specified
        for i in range(commits):
            test_file = f'feature_{stage.replace(" ", "_")}_{i}.txt'
            with open(test_file, 'w') as f:
                f.write(f'Content for {stage}')
            subprocess.run(['git', 'add', test_file], capture_output=True, check=True)
            subprocess.run(['git', 'commit', '-m', f'{stage} commit {i+1}'], capture_output=True, check=True)

        # Create tag if specified
        if tag:
            subprocess.run(['git', 'tag', tag], capture_output=True, check=True)

        # Get version information
        version = str(version_from_git(root=temp_dir, **PIPELINE_TEMPLATES))

        # Determine build status
        if tag and '-rc' not in tag:
            status = "✅ PRODUCTION"
        elif tag and '-rc' in tag:
            status = "🔄 RELEASE CANDIDATE"
        else:
            status = "🚧 DEVELOPMENT"

        print(f"\n[{stage}]")
        print(f"  Version: {version}")
        print(f"  Tag: {tag if tag else 'None'}")
        print(f"  Status: {status}")

        # Show version characteristics
        characteristics = []
        if 'dev' in version:
            characteristics.append("development build")
        if '+' in version:
            characteristics.append("includes commit hash")
        if 'dirty' in version:
            characteristics.append("uncommitted changes")
        if tag and tag in version:
            characteristics.append("tagged release")

        if characteristics:
            print(f"  Characteristics: {', '.join(characteristics)}")

    # Final summary
    print("\n" + "=" * 60)
    print("Production Package Summary:")
    print("=" * 60)

    final_version = str(version_from_git(root=temp_dir, **PIPELINE_TEMPLATES))
    all_tags = subprocess.run(['git', 'tag', '-l'], capture_output=True, text=True, check=True)
    tag_list = [t for t in all_tags.stdout.strip().split('\n') if t]

    print(f"  Final version: {final_version}")
    print(f"  Total tags: {len(tag_list)}")
    print(f"  Tags: {', '.join(tag_list)}")
    print(f"  ✅ Package ready for distribution!")

    # Show how version appears in different scenarios
    print("\n📊 Version format examples:")
    version_examples = [
        ("Tagged release", "1.0.0", "Matches git tag exactly"),
        ("Development build", "1.0.0.dev3+a1b2c3d", "Includes commit count and hash"),
        ("Dirty working tree", "1.0.0.dev3+a1b2c3d.dirty", "Shows uncommitted changes"),
    ]
    for scenario, example, description in version_examples:
        print(f"  • {scenario}: {example}")
        print(f"    {description}")

    # Show how to verify version
    print("\n📦 To verify version in production:")
    print("  $ python -m setuptools_git_versioning")
    print("  $ python -c 'from setuptools_git_versioning import version_from_git; print(version_from_git(root=\".\"))'")

os.chdir(tempfile.gettempdir())
print("\n✅ Production setup and CI/CD integration demonstrated")